<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [ ]:
%pip install langchain langchain_community pypdf
%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv
%pip install langchain-text-splitters

In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import warnings
warnings.filterwarnings('ignore')


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [9]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [10]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [22]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

In [23]:
load_dotenv()

True

In [24]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [27]:
from langchain_community.vectorstores import Chroma

In [28]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [29]:
user_question = "What practical implications for those who develop AI systems (such as data scientists and machine learning engineers) are discussed in AI for Everyone? Critical Perspectives in terms of ethics, responsibility, and social impact?" # User question
retrieved_docs = db.similarity_search(user_question, k=5) # k is the number of documents to retrieve

In [30]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
uses that essentially have technological solutions, preferably 
through further data collection and algorithmic sophistication. We see this for 
example with the growing industry that now concerns itself with ‘fairness’ in 
the design of systems, creating more inclusive data-sets and algorithms that 
can account for more diverse experiences, or the development of ‘bias mitiga-
tion’ tools (Zelevansky 2019). Such projects have drawn attention to some of 
the contentious assumptions that are embedded in the design of technological 
systems, but have also been accused of advancing technical fixes that serve to 
legitimise the industry (Gangadharan and Niklas 2019). 
The growing debate surrounding ethical challenges and the bias of algorith-
mic processes has helped spur on an engagement with data-driven technologies 
as socio-technical systems that have an impact on people’s lives. Some of this is
Document 2:
 we also see in the US, the mere fact of AI ethics principles hav-
i

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [31]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [32]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [38]:
prompt = f"""
## SYSTEM ROLE
You are a knowledgeable and critical chatbot designed to analyze the book *AI for Everyone? Critical Perspectives* (University of Westminster Press, 2021)
Your answers must be based exclusively on the provided context extracted from this book, with a focus on ethics, power, inequalities, and the social implications of AI for practitioners


## USER QUESTION
The user has asked:
"{user_question}"


## CONTEXT
Here is the relevant content from the book:
'''
{formatted_context}
'''


## GUIDELINES
1. **Accuracy**:
   - Only use the content in the `CONTEXT` section to answer.
   - If the answer cannot be found, explicitly state: "The provided context does not contain this information."
   - Prioritize discussions of power, capitalism, inequalities, labour, data justice, AI ethics, and their implications for how AI systems are designed and deployed

2. **Perspective**:
   - Reflect the critical tone of the book: question hype, techno-solutionism, and purely corporate framings of “AI for everyone”
   - Highlight who benefits and who is harmed by AI systems, and how this relates to developers’ responsibilities

3. **Practicality for Developers**:
   - Translate theoretical or critical arguments into concrete implications for data scientists and machine learning engineers (e.g. data practices, model design, documentation, attention to labour conditions, governance, and accountability)
   - When possible, turn insights into actionable recommendations or principles for practice.

4. **Transparency**:
   - Refer to the book and, if available in the context, mention chapter titles or section headings instead of page numbers.
   - Do not speculate beyond the context or add external information.

5. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.


## TASK
1. Answer the user's question **directly** based only on the given context.
2. Explicitly connect the book's critical perspectives to the practical work of AI developers.
3. Provide the response in the following format:


## RESPONSE FORMAT
'''
# [Brief, Informative Title]

[Answer in clear, structured text, 3–6 short paragraphs.]

## Practical Implications for AI Developers
- List 5–10 concrete, actionable recommendations for AI developers.
- Each bulletpoint must be a specific action or practice, not a placeholder.


**Source**:
• *AI for Everyone? Critical Perspectives* (ed. Pieter Verdegem, 2021), relevant chapters as provided in the context.
'''
"""
print("Prompt constructed.")



Prompt constructed.


In [39]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [40]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.9,  # Increase creativity
    'max_tokens': 3000,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.3,  # Reduce repetition
    'presence_penalty': 0.4    # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [41]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [42]:
answer = completion.choices[0].message.content
print(answer)

# Ethical Responsibilities in AI Development

The book *AI for Everyone? Critical Perspectives* emphasizes the ethical implications and social responsibilities of those who develop AI systems, such as data scientists and machine learning engineers. A significant concern is the pervasive bias embedded within algorithmic processes that can reinforce existing inequalities. Developers must recognize that merely implementing technical solutions, such as bias mitigation tools, often oversimplifies the complex social dynamics at play, potentially legitimizing systemic issues instead of addressing them directly.

Furthermore, discussions on ethical AI highlight the necessity for inclusivity in data collection and algorithm design. The historical context shows a dominance of perspectives from predominantly white, middle-class males, which has led to the exclusion of marginalized communities. As AI becomes more integrated into daily life, developers are called to ensure that their work reflects 

# Ethical Responsibilities in AI Development

The book *AI for Everyone? Critical Perspectives* emphasizes the ethical implications and social responsibilities of those who develop AI systems, such as data scientists and machine learning engineers. A significant concern is the pervasive bias embedded within algorithmic processes that can reinforce existing inequalities. Developers must recognize that merely implementing technical solutions, such as bias mitigation tools, often oversimplifies the complex social dynamics at play, potentially legitimizing systemic issues instead of addressing them directly.

Furthermore, discussions on ethical AI highlight the necessity for inclusivity in data collection and algorithm design. The historical context shows a dominance of perspectives from predominantly white, middle-class males, which has led to the exclusion of marginalized communities. As AI becomes more integrated into daily life, developers are called to ensure that their work reflects diverse experiences and enables meaningful participation from all societal members.

In addition to fairness and transparency principles, developers must grapple with broader existential questions regarding the societal impact of their technologies. They should strive to develop AI systems that genuinely contribute to human well-being rather than simply serving corporate or state interests. By acknowledging these ethical dimensions, developers can better align their work with the principles of accountability and beneficial outcomes for society.

## Practical Implications for AI Developers
- **Prioritize Inclusive Data Practices**: Actively seek out diverse datasets to ensure that AI models reflect varied perspectives and experiences.
- **Engage in Interdisciplinary Collaboration**: Work alongside ethicists, sociologists, and community representatives to address the socio-technical implications of AI systems.
- **Document Design Decisions**: Maintain transparent documentation detailing how design choices are made and the potential biases identified during development.
- **Adopt Ethical Guidelines**: Familiarize yourself with established ethical frameworks (e.g., Asilomar Principles) and incorporate them into project planning and execution.
- **Implement Bias Testing Procedures**: Develop formal methods to assess algorithmic bias regularly, akin to reliability and validity testing in traditional research.
- **Address Labor Conditions**: Advocate for fair labor practices within your teams and across supply chains involved in data gathering and model training.
- **Facilitate Public Engagement**: Create opportunities for community feedback on AI systems that affect public welfare, allowing for meaningful input in decision-making processes.
- **Explore Data Justice Principles**: Ensure that users are compensated fairly for their data contributions when applicable, fostering a sense of ownership over personal information.
- **Critically Assess Technical Solutions**: Resist techno-solutionism by questioning whether proposed technological fixes genuinely address deeper societal issues or merely serve as temporary patches.
- **Commit to Continuous Learning**: Stay informed about ongoing debates regarding ethics in AI and adapt practices accordingly based on emerging research and community needs.

**Source**:
• *AI for Everyone? Critical Perspectives* (ed. Pieter Verdegem, 2021), relevant chapters as provided in the context.

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [45]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [46]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [48]:
query_keywords = [
    "developers",
    "engineers",
    "labour",
    "workers",
    "governance",
    "data justice",
    "bias",
    "discrimination",
    "ethics",
    "power",
    "inequalities"
]
 # add your keywords
for i, doc in enumerate(retrieved_docs[:]):
    snippet = doc.page_content[:]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
274 AI for Everyone?
application; causes that essentially have technological solutions, preferably 
through further data collection and algorithmic sophistication. We see this for 
example with the growing industry that now concerns itself with ‘fairness’ in 
the design of systems, creating more inclusive data-sets and algorithms that 
can account for more diverse experiences, or the development of ‘bias mitiga-
tion’ tools (Zelevansky 2019). Such projects have drawn attention to some of 
the contentious assumptions that are embedded in the design of technological 
systems, but have also been accused of advancing technical fixes that serve to 
legitimise the industry (Gangadharan and Niklas 2019). 
The growing debate surrounding ethical challenges and the bias of algorith-
mic processes has helped spur on an engagement with data-driven technologies 
as socio-technical systems that have an impact on people’s lives. Some of this is
---------------------------------------------

1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:
